# SymptoSense DDXPlus: symptom triage modeling

This notebook documents the reproducible classical-ML workflow. DDXPlus contains synthetic cases; results do not establish clinical performance. Run `python train_pipeline.py --train-rows 150000` from `ml-service` to rebuild the models and report. The official test split is only used after training/model selection.

## Problem and data source

Predict the dataset pathology from patient evidence plus age and sex, and model the dataset-defined severity proxy. This is not a medical diagnosis. DDXPlus is synthetic and released under CC BY 4.0. INITIAL_EVIDENCE is excluded because the app does not collect it.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
from scipy import sparse
sys.path.insert(0, str(Path.cwd()))
from ml_pipeline.ddxplus_data import DATA, load_metadata, load_splits, vocabulary, feature_schema, encode_rows
evidence_meta, conditions = load_metadata()
print('Evidence entries:', len(evidence_meta), 'pathologies:', len(conditions))
print(pd.read_csv(DATA / 'release_train_patients', nrows=5).columns.tolist())

## EDA and preprocessing

The CSV `EVIDENCES` field is parsed as a list; valued items use `_@_`. Binary and valued evidence are expanded using the evidence metadata schema. Symptom and antecedent features retain distinct prefixes. Sampling is seeded and stratified by PATHOLOGY. Sex is masked to unknown for 20% of training rows. Age bins match the app, assigning age 50 to elderly to resolve its overlapping range.

In [ ]:
frames, sample_sizes, split_overlap, source_paths = load_splits(150000)
print(sample_sizes)
print('Exact-record fingerprint overlaps:', split_overlap)
print('Training duplicate evidence patterns:', int(frames['train']['EVIDENCES'].duplicated().sum()))
print(frames['train'].head(10)[['AGE','SEX','PATHOLOGY','EVIDENCES']].to_string(index=False))
columns = feature_schema()
X_train, y_train, _ = encode_rows(frames['train'].head(1000), columns, mask_sex=True)
print('Sparse smoke matrix:', X_train.shape, 'nonzero:', X_train.nnz)

## Model training and tuning

The single training entry point compares Logistic Regression, Multinomial Naive Bayes, Random Forest, and XGBoost with five-fold stratified CV on the training sample. The two CV leaders receive randomized searches using training folds only; the official validation split chooses the final estimator. The test set is untouched until final evaluation. For a complete retrain, run the command above; the next cell reads the generated report.

In [ ]:
report_path = Path('../ml-service/reports/ml_report.json') if Path.cwd().name == 'notebooks' else Path('reports/ml_report.json')
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('CV comparison:', json.dumps(report['training']['comparison'], indent=2))
    print('Tuning:', json.dumps(report['training']['hyperparameters'], indent=2))
else:
    print('No report found yet. Run python train_pipeline.py --train-rows 150000 from ml-service.')

## Final held-out evaluation, baselines and robustness

The report contains accuracy, macro and weighted precision/recall/F1, top-3/top-5, one-vs-rest macro ROC-AUC, confusion matrix, per-class metrics, train/test gap, majority and nearest-neighbour baselines, calibration comparison, evidence-drop and wrong-evidence tests, and a noise-augmented model. These are dataset performance measures only.

In [ ]:
if 'report' in globals():
    print('Test metrics:', json.dumps(report['evaluation']['metrics'], indent=2))
    print('Top-k:', report['evaluation']['topk'])
    print('Baselines:', json.dumps(report['baselines'], indent=2))
    print('Noise robustness:', json.dumps(report['noise_robustness'], indent=2))
    print('Calibration:', json.dumps(report['calibration'], indent=2))

## Explainability and urgency limitations

Global feature attribution and local top-five contributions describe model associations, not causes. Urgency labels use DDXPlus pathology severity, whose scale direction is lower-number-is-more-severe; the classifier is not clinician validated. Red-flag overrides and the disease safety floors are project safety choices, not clinical guidelines. The rule engine remains the triage authority.

In [ ]:
if 'report' in globals():
    print('Top global features:', json.dumps(report['explainability']['global_importance'], indent=2))
    print('Severity mapping:', json.dumps(report['severity_table'], indent=2))
    print('Limitations:', json.dumps(report['limitations'], indent=2))